In [139]:
import os
import numpy as np 
import pandas as pd
import tensorflow
from tensorflow import keras

In [140]:
folder_path="/mnt/d/Datasets/utkface_aligned_cropped/UTKFace"

In [141]:
import zipfile
zip=zipfile.ZipFile("/mnt/d/Datasets/archive.zip")

In [142]:
age=[]
gender=[]
image_path=[]
for file in os.listdir(folder_path):
    age.append(int(file.split('_')[0]))
    gender.append(int(file.split('_')[1]))
    image_path.append(file)


In [143]:
df = pd.DataFrame({'age':age,'gender':gender,'img':image_path})

In [144]:
df

,age,gender,img
0,100,0,100_0_0_20170112213500903.jpg.chip.jpg
1,100,0,100_0_0_20170112215240346.jpg.chip.jpg
2,100,1,100_1_0_20170110183726390.jpg.chip.jpg
3,100,1,100_1_0_20170112213001988.jpg.chip.jpg
4,100,1,100_1_0_20170112213303693.jpg.chip.jpg
...,...,...,...
23703,9,1,9_1_3_20161220222856346.jpg.chip.jpg
23704,9,1,9_1_3_20170104222949455.jpg.chip.jpg
23705,9,1,9_1_4_20170103200637399.jpg.chip.jpg
23706,9,1,9_1_4_20170103200814791.jpg.chip.jpg


In [145]:
df.shape

(23708, 3)

In [146]:
train_df=df.sample(frac=1,random_state=2).iloc[:20000]
test_df = df.sample(frac=1,random_state=2).iloc[20000:]

In [147]:
train_df.shape

(20000, 3)

In [148]:
test_df.shape

(3708, 3)

In [149]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [150]:
train_datagen = ImageDataGenerator(rescale=1/255.0,
                                   rotation_range=30,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   shear_range=0.2,
                                   horizontal_flip=True
                                   )

In [151]:
test_datagen  = ImageDataGenerator(rescale=1/255.0)

In [152]:
train_generator= train_datagen.flow_from_dataframe(train_df,directory=folder_path,x_col='img',y_col=['age','gender'],target_size=(224,224),class_mode='multi_output')


Found 20000 validated image filenames.


In [153]:
test_generator=test_datagen.flow_from_dataframe(test_df,directory=folder_path,x_col='img',y_col=['age','gender'],target_size=(224,224),class_mode='multi_output')

Found 3708 validated image filenames.


In [168]:
import tensorflow as tf

def wrap_gen(keras_iterator):
    def gen():
        for x_batch, y_batch in keras_iterator:
            # y_batch is [age_array, gender_array] -> convert to tuple
            yield x_batch, (y_batch[0], y_batch[1])
    return gen

output_signature = (
    tf.TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32),   # images
    (
        tf.TensorSpec(shape=(None,), dtype=tf.float32),  # age
        tf.TensorSpec(shape=(None,), dtype=tf.float32),  # gender
    )
)

train_ds = tf.data.Dataset.from_generator(
    wrap_gen(train_generator),
    output_signature=output_signature
).repeat()

test_ds = tf.data.Dataset.from_generator(
    wrap_gen(test_generator),
    output_signature=output_signature
).repeat()

In [169]:
from keras.applications.resnet50 import ResNet50
from keras.layers import *
from keras.models import Model


In [170]:
resnet = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))


In [171]:
resnet.trainable=False

In [172]:
output = resnet.layers[-1].output


In [173]:
type(output)

keras.src.backend.common.keras_tensor.KerasTensor

In [174]:
flat = Flatten()(output)
dense1= Dense(512,activation='relu')(flat)
dense2 = Dense(512,activation='relu')(flat)
dense3 = Dense(256,activation='relu')(dense1)
dense4 = Dense(256,activation='relu')(dense2)
output1 = Dense(1,activation='linear',name='age')(dense3)
output2 = Dense(1,activation='sigmoid',name='gender')(dense4)

In [ ]:
output = resnet.layers[-1].output

flatten = Flatten()(output)

dense1 = Dense(512, activation='relu')(flatten)
dense2 = Dense(512,activation='relu')(flatten)

dense3 = Dense(512,activation='relu')(dense1)
dense4 = Dense(512,activation='relu')(dense2)

output1 = Dense(1,activation='linear',name='age')(dense3)
output2 = Dense(1,activation='sigmoid',name='gender')(dense4)

In [176]:
model = Model(inputs=resnet.input,outputs=[output1,output2])

In [177]:
model.compile(optimizer='adam',loss={'age':'mae','gender':'binary_crossentropy'},loss_weights={'age':1,'gender':100})

In [178]:
model.fit(
    train_ds,
    epochs=10,
    steps_per_epoch=len(train_generator),
    validation_data=test_ds,
    validation_steps=len(test_generator)
)

Epoch 1/10


I0000 00:00:1788246523.905443    6085 service.cc:153] XLA service 0x7eb8c0035d50 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1788246523.905525    6085 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 5050 Laptop GPU, Compute Capability 12.0a (Driver: 13.2.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.24.0)
I0000 00:00:1788246524.185864    6085 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1788246525.764752    6085 cuda_dnn.cc:461] Loaded cuDNN version 92400
I0000 00:00:1788246525.877833    6085 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_45934__.180
I0000 00:00:1788246530.612849   10190 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_24', 8 bytes spill stores, 8 bytes spill loads

I0000 00:00:1788246531.677494   10183 subprocess_compilation.cc:348

  1/625 ━━━━━━━━━━━━━━━━━━━━ 5:55:38 34s/step - age_loss: 29.1009 - gender_loss: 0.8830 - loss: 117.4031

I0000 00:00:1788246553.276829    6085 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 274ms/step - age_loss: 15.3579 - gender_loss: 0.8315 - loss: 98.5106

I0000 00:00:1788246743.178602   10762 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_16', 4 bytes spill stores, 4 bytes spill loads

I0000 00:00:1788246744.919037   10777 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_12', 8 bytes spill stores, 8 bytes spill loads

W0000 00:00:1788246758.007793    6085 bfc_allocator.cc:502] Allocator (GPU_0_bfc) ran out of memory trying to allocate 630.86MiB (rounded to 661504768)requested by op 
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
I0000 00:00:1788246758.007883    6085 bfc_allocator.cc:1049] BFCAllocator dump for GPU_0_bfc
I0000 00:00:1788246758.007885    6085 bfc_allocator.cc:1056] Bin (256): 	Total Chunks: 220, Chunks in use: 220. 55.0KiB allocat

625/625 ━━━━━━━━━━━━━━━━━━━━ 251s 347ms/step - age_loss: 15.3579 - gender_loss: 0.8315 - loss: 98.5106 - val_age_loss: 15.8756 - val_gender_loss: 0.6923 - val_loss: 85.1131
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 182s 292ms/step - age_loss: 14.8608 - gender_loss: 0.6940 - loss: 84.2628 - val_age_loss: 14.5965 - val_gender_loss: 0.6918 - val_loss: 83.7763
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 225s 361ms/step - age_loss: 14.7232 - gender_loss: 0.6956 - loss: 84.2879 - val_age_loss: 14.4877 - val_gender_loss: 0.6920 - val_loss: 83.6884
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 227s 364ms/step - age_loss: 14.5861 - gender_loss: 0.6931 - loss: 83.9003 - val_age_loss: 14.2235 - val_gender_loss: 0.6923 - val_loss: 83.4563
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1675s 3s/step - age_loss: 14.6521 - gender_loss: 0.6927 - loss: 83.9195 - val_age_loss: 14.1795 - val_gender_loss: 0.6927 - val_loss: 83.4508
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 178s 285ms/step - age_loss: 14.4334 - gender_loss: 0.

In [179]:
import cv2


ModuleNotFoundError: No module named 'cv2'

In [8]:
model.save("age_gender_model.keras")

In [ ]:
from keras.models import load_model

In [10]:
model = load_model("/mnt/d/DL-Algorithm/functional-API/age_gender_model.keras")

In [11]:
import cv2 as cv

In [12]:
cap = cv.VideoCapture(0)
if not cap.isOpened():
    print("Error: Camera is not opening")
else:
    while True:
        ret,frame = cap.read()
        if not ret:
            print("Could Not read frame")
            break

Error: Camera is not opening


[ WARN:0@1354.334] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video0): can't open camera by index
[ WARN:0@1354.338] global cap.cpp:435 open VIDEOIO(FFMPEG): raised OpenCV exception:

OpenCV(5.0.0) /io/opencv/modules/videoio/src/cap_ffmpeg_impl.hpp:1243: error: (-2:Unspecified error) in function 'bool CvCapture_FFMPEG::open(const char*, int, const cv::Ptr<cv::IStreamReader>&, const cv::VideoCaptureParameters&)'
> VIDEOIO/FFMPEG: Camera index out of range (expected: 'index < device_list->nb_devices'), where
>     'index' is 0
> must be less than
>     'device_list->nb_devices' is 0


[ERROR:0@1354.339] global obsensor_uvc_stream_channel.cpp:163 getStreamChannelGroup Camera index out of range
